Regresión Lineal Multivariante

In [1]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
# Variables exógenas que se usarán como features
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
# Features de tiempo para capturar ciclicidades
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# Lags para las features del modelo de predicción
N_LAGS_AQI = 60  # Lags para la variable objetivo (AQI)
N_LAGS_EXO = 60  # Lags para las variables exógenas (contaminantes)

# Horizontes de predicción actualizados
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5 # Splits para validación cruzada cronológica

# --- Rutas (Asegúrate de que coincidan con tu estructura) ---
PROJECT_ROOT_DIR = os.getcwd()

ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')

# Directorios de datos de entrada
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')

# Directorio de salida para las métricas de este modelo
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
# Nuevo nombre para reflejar el modelo multivariante
PREDICTION_METHOD_NAME = 'regresionLinealMultipleLags' 
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares (Modificadas para Multivariante y Eficiencia) ---

def create_multivariate_lagged_features(df_full_city_data, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon):
    """
    Crea features multivariantes de manera eficiente para evitar la fragmentación del DataFrame.
    """
    # Usar una lista para recolectar todas las nuevas columnas (como Series)
    features_list = []

    # Lags de la variable objetivo
    for i in range(1, n_lags_target + 1):
        lagged_series = df_full_city_data[target_col].shift(i)
        lagged_series.name = f'{target_col}_lag{i}'
        features_list.append(lagged_series)

    # Lags de variables exógenas
    for exo_col in exo_cols:
        if exo_col == target_col: continue
        for i in range(1, n_lags_exo + 1):
            lagged_series = df_full_city_data[exo_col].shift(i)
            lagged_series.name = f'{exo_col}_lag{i}'
            features_list.append(lagged_series)
    
    # Features de tiempo (contemporáneas, t)
    for tf_col in time_features_cols:
        if tf_col in df_full_city_data.columns:
            features_list.append(df_full_city_data[tf_col])
            
    # Combinar todas las features en un solo DataFrame
    df_features = pd.concat(features_list, axis=1)
    
    # Crear el target
    df_target = pd.DataFrame(df_full_city_data[target_col].shift(-horizon))
    df_target.columns = ['target_h']
    
    # Unir features y target por el índice
    df_processed = pd.concat([df_features, df_target], axis=1)
    
    # Eliminar filas con cualquier NaN generado por los shifts
    df_processed = df_processed.dropna()
    
    if df_processed.empty:
        return pd.DataFrame(), pd.Series(dtype='float64')
    
    y = df_processed['target_h']
    X = df_processed.drop(columns=['target_h'])
    return X, y

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_predict_eval_multi_horizon(df_city_data_input, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizons_list, n_cv_splits):
    """
    Entrena, predice y evalúa un modelo multivariante para múltiples horizontes usando TimeSeriesSplit.
    """
    results_by_horizon = []
    max_lags = max(n_lags_target, n_lags_exo)
    min_len = max_lags + max(horizons_list) + n_cv_splits * 10
    
    if len(df_city_data_input) < min_len:
        print(f"    Datos insuficientes. Mínimo requerido: {min_len}, disponible: {len(df_city_data_input)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon

    for horizon in horizons_list:
        X, y = create_multivariate_lagged_features(
            df_city_data_input, target_col, exo_cols, time_features_cols,
            n_lags_target, n_lags_exo, horizon
        )

        if X.empty or y.empty or len(X) < n_cv_splits * 2:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        horizon_maes, horizon_rmses, horizon_mases = [], [], []

        for fold, (train_index, test_index) in enumerate(tscv.split(X)):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            if X_train.empty or y_train.empty or X_test.empty or y_test.empty or len(y_train) <=1:
                continue

            model = LinearRegression()
            try:
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                original_series_train_end_idx = X_train.index[-1]
                y_train_for_naive = df_city_data_input[target_col].iloc[:original_series_train_end_idx + 1].values
                
                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                mase = calculate_mase(y_test.values, predictions, y_train_for_naive)

                horizon_maes.append(mae)
                horizon_rmses.append(rmse)
                if not np.isnan(mase): horizon_mases.append(mase)
            except Exception as e:
                print(f"        Error en Fold {fold+1} para horizonte {horizon}h: {e}")
                continue
        
        results_by_horizon.append({
            'Horizon': horizon,
            'MAE': np.nanmean(horizon_maes) if horizon_maes else np.nan,
            'RMSE': np.nanmean(horizon_rmses) if horizon_rmses else np.nan,
            'MASE': np.nanmean(horizon_mases) if horizon_mases else np.nan
        })
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

# 1. Cargar y preprocesar DataFrame Original (Baseline)
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'.")
    df_original_full = None

if df_original_full is not None:
    print("\n--- Predicción Multivariante con Datos Originales (Baseline) ---")
    for city_name in df_original_full['City'].unique():
        print(f"  Ciudad: {city_name}")
        df_city_original = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        
        # Rellenar NaNs en el baseline de forma robusta
        cols_to_fill = [TARGET_VARIABLE] + EXOGENOUS_COLS
        for col in cols_to_fill:
            if col in df_city_original.columns and df_city_original[col].isnull().any():
                df_city_original[col] = df_city_original[col].ffill().bfill()
                if df_city_original[col].isnull().any():
                    global_mean = df_original_full[col].mean()
                    df_city_original[col] = df_city_original[col].fillna(global_mean)

        if df_city_original[cols_to_fill].isnull().values.any():
            print(f"    ADVERTENCIA: No se pudieron rellenar todos los NaNs en el baseline para {city_name}. Omitiendo ciudad.")
            continue
        
        horizon_metrics = train_predict_eval_multi_horizon(
            df_city_original, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
            N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Imputation_Method': 'Original_FFill_Mean', 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        if horizon_metrics:
            if not np.isnan(horizon_metrics[0]['MAE']):
                print(f"    Métricas (Baseline, {horizon_metrics[0]['Horizon']}h): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}, MASE={horizon_metrics[0]['MASE']:.4f}")
            if len(horizon_metrics) > 1 and not np.isnan(horizon_metrics[-1]['MAE']):
                print(f"    Métricas (Baseline, {horizon_metrics[-1]['Horizon']}h): MAE={horizon_metrics[-1]['MAE']:.4f}, RMSE={horizon_metrics[-1]['RMSE']:.4f}, MASE={horizon_metrics[-1]['MASE']:.4f}")

# 2. Iterar sobre todos los métodos de imputación y sus archivos
print("\n--- Predicción Multivariante con Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]

for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root):
        continue
    
    print(f"\n--- Procesando datos del método de imputación: {method_name} ---")
    
    for city_folder in os.listdir(imputed_csvs_root):
        city_path = os.path.join(imputed_csvs_root, city_folder)
        if not os.path.isdir(city_path): continue
        
        original_city_name = city_folder.replace("_", " ")
        print(f"  Ciudad: {original_city_name}")

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")

            repetition_files = glob.glob(os.path.join(scenario_path, "*.csv"))
            if not repetition_files: continue

            scenario_metrics_accumulator = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}

            for rep_file_path in repetition_files:
                file_name = os.path.basename(rep_file_path)
                rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                
                try:
                    df_rep = pd.read_csv(rep_file_path)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                    df_rep['Hour'] = df_rep['Date'].dt.hour
                    df_rep['DayOfWeek'] = df_rep['Date'].dt.dayofweek
                    df_rep['Month'] = df_rep['Date'].dt.month
                except Exception as e:
                    print(f"        Error cargando o procesando {file_name}: {e}. Omitiendo.")
                    continue
                
                df_city_rep = df_rep[df_rep['City'] == original_city_name].copy().reset_index(drop=True)

                if df_city_rep.isnull().values.any():
                    print(f"        ADVERTENCIA: NaNs encontrados en {file_name} DESPUÉS de imputar. Rellenando con ffill/bfill.")
                    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
                        if col in df_city_rep.columns and df_city_rep[col].isnull().any():
                             df_city_rep[col] = df_city_rep[col].ffill().bfill()
                    if df_city_rep.isnull().values.any():
                        print(f"        ERROR: NaNs persisten después de ffill/bfill en {file_name}. Omitiendo archivo.")
                        continue
                
                horizon_metrics_rep = train_predict_eval_multi_horizon(
                    df_city_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
                    N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS
                )

                for hm_rep in horizon_metrics_rep:
                    h = hm_rep['Horizon']
                    all_prediction_metrics_detailed.append({
                        'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': h,
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })
                    if not np.isnan(hm_rep.get('MAE')): scenario_metrics_accumulator[h]['maes'].append(hm_rep['MAE'])
                    if not np.isnan(hm_rep.get('RMSE')): scenario_metrics_accumulator[h]['rmses'].append(hm_rep['RMSE'])
                    if not np.isnan(hm_rep.get('MASE')): scenario_metrics_accumulator[h]['mases'].append(hm_rep['MASE'])

            print(f"      Promedios para Escenario '{scenario_folder}':")
            for h_summary in [PREDICTION_HORIZONS[0], PREDICTION_HORIZONS[-1]]:
                if h_summary in scenario_metrics_accumulator:
                    avg_mae = np.nanmean(scenario_metrics_accumulator[h_summary]['maes']) if scenario_metrics_accumulator[h_summary]['maes'] else np.nan
                    avg_rmse = np.nanmean(scenario_metrics_accumulator[h_summary]['rmses']) if scenario_metrics_accumulator[h_summary]['rmses'] else np.nan
                    avg_mase = np.nanmean(scenario_metrics_accumulator[h_summary]['mases']) if scenario_metrics_accumulator[h_summary]['mases'] else np.nan
                    if not np.isnan(avg_mae):
                        print(f"        Horizonte {h_summary}h: MAE={avg_mae:.4f}, RMSE={avg_rmse:.4f}, MASE={avg_mase:.4f}")

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)

    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\nTodas las métricas de predicción detalladas guardadas en: {detailed_output_path}")
    print(df_all_metrics.head())

    df_summary = df_all_metrics.groupby(
        ['City', 'Imputation_Method', 'Scenario', 'Horizon_h']
    ).agg(
        Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
        Count=('MAE', 'count')
    ).reset_index()

    for city_name in df_summary['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary[df_summary['City'] == city_name]
        
        city_metrics_dir = os.path.join(PREDICTION_METRICS_ROOT_DIR, city_name_safe)
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_safe}.csv')
        
        df_city_summary.to_csv(city_summary_path, index=False)
        print(f"  Resumen de métricas para {city_name} guardado en: {city_summary_path}")

    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(
        Global_Avg_MAE=('MAE', 'mean'),
        Global_Avg_RMSE=('RMSE', 'mean'),
        Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"\nResumen global de métricas promedio guardado en: {global_summary_path}")
    print("--- Resumen Global por Método y Horizonte (promedio de ciudades y escenarios) ---")
    print(df_global_summary)
    
    # --- INICIO DE LA NUEVA SECCIÓN ---
    # Calcular el promedio de las métricas a través de todos los escenarios para cada método
    
    # Excluir el 'Scenario' == 'N/A' del baseline para no distorsionar el promedio
    df_metrics_for_avg = df_all_metrics[df_all_metrics['Scenario'] != 'N/A'].copy()

    df_avg_by_method = df_metrics_for_avg.groupby(
        ['Imputation_Method', 'Horizon_h']
    ).agg(
        Avg_Across_Scenarios_MAE=('MAE', 'mean'),
        Avg_Across_Scenarios_RMSE=('RMSE', 'mean'),
        Avg_Across_Scenarios_MASE=('MASE', 'mean')
    ).reset_index()

    # Añadir el baseline de vuelta al resumen si existe
    df_baseline = df_all_metrics[df_all_metrics['Imputation_Method'] == 'Original_FFill_Mean'].copy()
    if not df_baseline.empty:
        df_baseline_summary = df_baseline.groupby(['Imputation_Method', 'Horizon_h']).agg(
            Avg_Across_Scenarios_MAE=('MAE', 'mean'),
            Avg_Across_Scenarios_RMSE=('RMSE', 'mean'),
            Avg_Across_Scenarios_MASE=('MASE', 'mean')
        ).reset_index()
        df_avg_by_method = pd.concat([df_avg_by_method, df_baseline_summary], ignore_index=True).sort_values(by=['Imputation_Method', 'Horizon_h'])

    # Guardar el nuevo CSV
    avg_by_method_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_avg_by_method_{PREDICTION_METHOD_NAME}.csv')
    df_avg_by_method.to_csv(avg_by_method_path, index=False)
    print(f"\nResumen de métricas promediadas por método guardado en: {avg_by_method_path}")
    print("--- Resumen por Método (promedio de ciudades, escenarios y repeticiones) ---")
    print(df_avg_by_method)
    # --- FIN DE LA NUEVA SECCIÓN ---


script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

DataFrame original 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

--- Predicción Multivariante con Datos Originales (Baseline) ---
  Ciudad: Brasilia
    Métricas (Baseline, 12h): MAE=5.7678, RMSE=7.3499, MASE=3.4510
    Métricas (Baseline, 60h): MAE=10.2093, RMSE=12.8170, MASE=6.1020
  Ciudad: Cairo
    Métricas (Baseline, 12h): MAE=6.4057, RMSE=7.9000, MASE=6.7811
    Métricas (Baseline, 60h): MAE=13.7959, RMSE=17.0310, MASE=14.5781
  Ciudad: Dubai
    Métricas (Baseline, 12h): MAE=6.9764, RMSE=9.0235, MASE=8.0043
    Métricas (Baseline, 60h): MAE=22.2569, RMSE=27.0626, MASE=25.3508
  Ciudad: London
    Métricas (Baseline, 12h): MAE=4.9318, RMSE=6.2782, MASE=4.1129
    Métricas (Baseline, 60h): MAE=7.6345, RMSE=10.5137, MASE=6.2679
  Ciudad: New York
    Métricas (Baseline, 12h): MAE=7.2511, RMSE=9.0380, MASE=4.5129
    Métricas (Baseline, 60h): MAE=13.6887, RMSE=16.5922, MASE=8.2775
  Ciudad: Sydney
    Métricas (Baseline, 12h): MAE=5.

SARIMAX

In [5]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month'] # Se añadirán como variables exógenas

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5 # Usar más splits para una evaluación más robusta

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
# Directorio donde se guardaron los órdenes del script anterior
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', 'SARIMA', 'model_orders')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'SARIMAX_pred' # Nuevo nombre para el modelo
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_predict_eval_sarimax_multi_horizon(
    df_city_data_input, target_col, exo_cols, time_features_cols,
    sarima_order, sarima_seasonal_order, horizons_list, n_cv_splits
):
    """
    Entrena, predice y evalúa un modelo SARIMAX multivariante para múltiples horizontes.
    """
    results_by_horizon = []
    
    # Preparar datos: endog (y) y exog (X)
    y = df_city_data_input[target_col]
    X = df_city_data_input[exo_cols + time_features_cols]
    
    # Comprobar si hay suficientes datos para la validación cruzada
    min_len = (sum(sarima_order) + sum(sarima_seasonal_order)) * 2 + max(horizons_list) + n_cv_splits
    if len(y) < min_len:
        print(f"    Datos insuficientes. Mínimo requerido: ~{min_len}, disponible: {len(y)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon
        
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons_list))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}

    # NUEVO PRINT: Informar sobre el inicio del bucle de validación cruzada
    print(f"      -> Iniciando validación cruzada con {n_cv_splits} folds...")
    for fold, (train_index, test_index) in enumerate(tscv.split(X)):
        # NUEVO PRINT: Informar sobre el fold actual
        print(f"        -> Fold {fold + 1}/{n_cv_splits}...")
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]

        if y_train.empty or X_train.empty:
            print("        -> Fold omitido por datos vacíos en train set.")
            continue

        try:
            # NUEVO PRINT: Informar sobre el ajuste del modelo
            print("          -> Ajustando modelo SARIMAX...")
            model = SARIMAX(y_train, exog=X_train, order=sarima_order, seasonal_order=sarima_seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            results_fit = model.fit(disp=False, maxiter=200)

            # NUEVO PRINT: Informar sobre la predicción
            print("          -> Realizando predicciones...")
            forecasts = results_fit.get_forecast(steps=max(horizons_list), exog=X_test.head(max(horizons_list)))
            forecast_values = forecasts.predicted_mean

            for h in horizons_list:
                y_true_h = y_test.head(h).values
                y_pred_h = forecast_values.head(h).values

                if len(y_true_h) < h: continue

                mae = mean_absolute_error(y_true_h, y_pred_h)
                rmse = np.sqrt(mean_squared_error(y_true_h, y_pred_h))
                mase = calculate_mase(y_true_h, y_pred_h, y_train.values)
                
                accumulated_metrics[h]['maes'].append(mae)
                accumulated_metrics[h]['rmses'].append(rmse)
                if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)

        except Exception as e:
            print(f"        -> Error en Fold {fold+1}: {e}")
            continue
    
    for h in horizons_list:
        results_by_horizon.append({
            'Horizon': h,
            'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan,
            'RMSE': np.nanmean(accumulated_metrics[h]['rmses']) if accumulated_metrics[h]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan
        })
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

# 1. Cargar y preprocesar DataFrame Original (Baseline)
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'], utc=True)
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'.")
    df_original_full = None

if df_original_full is not None:
    print("\n--- Predicción SARIMAX con Datos Originales (Baseline) ---")
    for city_name in df_original_full['City'].unique():
        print(f"  Procesando ciudad (Baseline): {city_name}")
        df_city_original = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        
        cols_to_fill = [TARGET_VARIABLE] + EXOGENOUS_COLS
        for col in cols_to_fill:
            if col in df_city_original.columns and df_city_original[col].isnull().any():
                df_city_original[col] = df_city_original[col].interpolate(method='linear').ffill().bfill()
        if df_city_original[cols_to_fill].isnull().values.any():
            print(f"    ADVERTENCIA: No se pudieron rellenar todos los NaNs para {city_name}. Omitiendo.")
            continue
        
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
        try:
            with open(order_file_path, 'r') as f: orders = json.load(f)
            order_tuple = tuple(orders['order']); seasonal_order_tuple = tuple(orders['seasonal_order'])
            print(f"    -> Órdenes cargadas para {city_name}: {order_tuple}{seasonal_order_tuple}")
        except Exception:
            print(f"    -> Órdenes no encontradas en {order_file_path}. Usando (1,1,1)(1,1,0,24).")
            order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)
        
        horizon_metrics = train_predict_eval_sarimax_multi_horizon(
            df_city_original, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
            order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Imputation_Method': 'Original_Interpolate', 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        print(f"    -> Finalizado Baseline para {city_name}.")


# 2. Iterar sobre todos los métodos de imputación y sus archivos
print("\n--- Predicción SARIMAX con Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]

for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Procesando método de imputación: {method_name} ---")
    
    for city_folder in os.listdir(imputed_csvs_root):
        city_path, original_city_name = os.path.join(imputed_csvs_root, city_folder), city_folder.replace("_", " ")
        if not os.path.isdir(city_path): continue
        print(f"  Procesando ciudad: {original_city_name}")
        
        city_name_safe = original_city_name.replace(" ", "_").replace("-", "_")
        order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
        try:
            with open(order_file_path, 'r') as f: orders = json.load(f)
            order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
            print(f"    -> Órdenes cargadas para {original_city_name}: {order_tuple}{seasonal_order_tuple}")
        except Exception:
            print(f"    -> Órdenes no encontradas para {city_name_safe}. Usando (1,1,1)(1,1,0,24).")
            order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Procesando escenario: {scenario_folder}")

            for rep_file_path in glob.glob(os.path.join(scenario_path, "*.csv")):
                file_name = os.path.basename(rep_file_path)
                # NUEVO PRINT: Informar sobre el archivo actual
                print(f"      -> Procesando archivo: {file_name}")
                rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                try:
                    df_rep = pd.read_csv(rep_file_path)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                    df_rep['Hour'] = df_rep['Date'].dt.hour
                    df_rep['DayOfWeek'] = df_rep['Date'].dt.dayofweek
                    df_rep['Month'] = df_rep['Date'].dt.month
                except Exception as e:
                    print(f"        -> Error cargando {file_name}: {e}. Omitiendo.")
                    continue
                
                if df_rep.isnull().values.any():
                    print(f"        -> ADVERTENCIA: NaNs en {file_name}. Rellenando con ffill/bfill.")
                    df_rep = df_rep.ffill().bfill()
                    if df_rep.isnull().values.any(): continue
                
                horizon_metrics_rep = train_predict_eval_sarimax_multi_horizon(
                    df_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
                    order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS
                )
                for hm_rep in horizon_metrics_rep:
                    all_prediction_metrics_detailed.append({
                        'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': hm_rep['Horizon'],
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    # 1. Guardar métricas detalladas
    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\n[OK] Todas las métricas detalladas guardadas en: {detailed_output_path}")

    # 2. Guardar resumen por ciudad y escenario
    df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(
        Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
        Count=('MAE', 'count')
    ).reset_index()
    summary_by_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_summary_by_city.to_csv(summary_by_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_by_city_path}")

    # 3. Guardar resumen global por método y horizonte (promediando ciudades y escenarios)
    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(
        Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
    print("\n--- Resumen Global por Método y Horizonte ---")
    print(df_global_summary)

    # 4. NUEVO: Guardar resumen promediando horizontes para cada método
    df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(
        Final_Avg_MAE=('Global_Avg_MAE', 'mean'),
        Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'),
        Final_Avg_MASE=('Global_Avg_MASE', 'mean')
    ).reset_index()
    avg_over_horizons_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    df_avg_over_horizons.to_csv(avg_over_horizons_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {avg_over_horizons_path}")
    print("\n--- Resumen Final por Método (promedio de todas las ciudades, escenarios y horizontes) ---")
    print(df_avg_over_horizons)

script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

DataFrame original 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

--- Predicción SARIMAX con Datos Originales (Baseline) ---
  Procesando ciudad (Baseline): Brasilia
    -> Órdenes cargadas para Brasilia: (1, 0, 0)(1, 0, 1, 24)
      -> Iniciando validación cruzada con 5 folds...
        -> Fold 1/5...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
        -> Fold 2/5...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
        -> Fold 3/5...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
        -> Fold 4/5...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
        -> Fold 5/5...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
    -> Finalizado Baseline para Brasilia.
  Procesando ciudad (Baseline): Cairo
    -> Órdenes cargadas para Cairo: (2, 1, 0)(0, 0, 0, 24)
      -> Ini

KeyboardInterrupt: 

In [7]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month'] # Se añadirán como variables exógenas

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- MODIFICACIONES PARA PRUEBAS RÁPIDAS ---
TARGET_CITY_FILTER = "Brasilia" # Poner a None para procesar todas las ciudades
N_REPETITIONS_TO_PROCESS = 1    # Número de repeticiones a procesar por escenario
N_CV_SPLITS = 2                 # Reducido para una evaluación más rápida
# -------------------------------------------

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
# Directorio donde se guardaron los órdenes del script anterior
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', 'SARIMA', 'model_orders')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'SARIMAX_pred' # Nuevo nombre para el modelo
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_predict_eval_sarimax_multi_horizon(
    df_city_data_input, target_col, exo_cols, time_features_cols,
    sarima_order, sarima_seasonal_order, horizons_list, n_cv_splits
):
    """
    Entrena, predice y evalúa un modelo SARIMAX multivariante para múltiples horizontes.
    """
    results_by_horizon = []
    
    # Preparar datos: endog (y) y exog (X)
    y = df_city_data_input[target_col]
    X = df_city_data_input[exo_cols + time_features_cols]
    
    # Comprobar si hay suficientes datos para la validación cruzada
    min_len = (sum(sarima_order) + sum(sarima_seasonal_order)) * 2 + max(horizons_list) + n_cv_splits
    if len(y) < min_len:
        print(f"    Datos insuficientes. Mínimo requerido: ~{min_len}, disponible: {len(y)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon
        
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons_list))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}

    print(f"      -> Iniciando validación cruzada con {n_cv_splits} fold(s)...")
    for fold, (train_index, test_index) in enumerate(tscv.split(X)):
        print(f"        -> Fold {fold + 1}/{n_cv_splits}...")
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]

        if y_train.empty or X_train.empty:
            print("        -> Fold omitido por datos vacíos en train set.")
            continue

        try:
            print("          -> Ajustando modelo SARIMAX...")
            model = SARIMAX(y_train, exog=X_train, order=sarima_order, seasonal_order=sarima_seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            results_fit = model.fit(disp=False, maxiter=200)

            print("          -> Realizando predicciones...")
            forecasts = results_fit.get_forecast(steps=max(horizons_list), exog=X_test.head(max(horizons_list)))
            forecast_values = forecasts.predicted_mean

            for h in horizons_list:
                y_true_h = y_test.head(h).values
                y_pred_h = forecast_values.head(h).values

                if len(y_true_h) < h: continue

                mae = mean_absolute_error(y_true_h, y_pred_h)
                rmse = np.sqrt(mean_squared_error(y_true_h, y_pred_h))
                mase = calculate_mase(y_true_h, y_pred_h, y_train.values)
                
                accumulated_metrics[h]['maes'].append(mae)
                accumulated_metrics[h]['rmses'].append(rmse)
                if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)

        except Exception as e:
            print(f"        -> Error en Fold {fold+1}: {e}")
            continue
    
    for h in horizons_list:
        results_by_horizon.append({
            'Horizon': h,
            'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan,
            'RMSE': np.nanmean(accumulated_metrics[h]['rmses']) if accumulated_metrics[h]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan
        })
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

# 1. Cargar y preprocesar DataFrame Original (Baseline)
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'], utc=True)
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'.")
    df_original_full = None

# Determinar la lista de ciudades a procesar
if TARGET_CITY_FILTER:
    cities_to_process = [TARGET_CITY_FILTER]
    print(f"\nATENCIÓN: Se procesará únicamente la ciudad: {TARGET_CITY_FILTER}")
else:
    cities_to_process = df_original_full['City'].unique()
    print("\nATENCIÓN: Se procesarán todas las ciudades encontradas en los datos.")

if df_original_full is not None:
    print("\n--- Predicción SARIMAX con Datos Originales (Baseline) ---")
    for city_name in cities_to_process:
        if city_name not in df_original_full['City'].unique(): continue
        
        print(f"  Procesando ciudad (Baseline): {city_name}")
        df_city_original = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        
        cols_to_fill = [TARGET_VARIABLE] + EXOGENOUS_COLS
        for col in cols_to_fill:
            if col in df_city_original.columns and df_city_original[col].isnull().any():
                df_city_original[col] = df_city_original[col].interpolate(method='linear').ffill().bfill()
        if df_city_original[cols_to_fill].isnull().values.any():
            print(f"    ADVERTENCIA: No se pudieron rellenar todos los NaNs para {city_name}. Omitiendo.")
            continue
        
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
        try:
            with open(order_file_path, 'r') as f: orders = json.load(f)
            order_tuple = tuple(orders['order']); seasonal_order_tuple = tuple(orders['seasonal_order'])
            print(f"    -> Órdenes cargadas para {city_name}: {order_tuple}{seasonal_order_tuple}")
        except Exception:
            print(f"    -> Órdenes no encontradas en {order_file_path}. Usando (1,1,1)(1,1,0,24).")
            order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)
        
        horizon_metrics = train_predict_eval_sarimax_multi_horizon(
            df_city_original, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
            order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Imputation_Method': 'Original_Interpolate', 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        print(f"    -> Finalizado Baseline para {city_name}.")


# 2. Iterar sobre todos los métodos de imputación y sus archivos
print("\n--- Predicción SARIMAX con Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]

for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Procesando método de imputación: {method_name} ---")
    
    # Filtrar para procesar solo la ciudad objetivo si está definida
    city_folders_to_process = [c for c in os.listdir(imputed_csvs_root) if (TARGET_CITY_FILTER is None or c == TARGET_CITY_FILTER.replace(" ", "_"))]

    for city_folder in city_folders_to_process:
        city_path, original_city_name = os.path.join(imputed_csvs_root, city_folder), city_folder.replace("_", " ")
        if not os.path.isdir(city_path): continue
        print(f"  Procesando ciudad: {original_city_name}")
        
        city_name_safe = original_city_name.replace(" ", "_").replace("-", "_")
        order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
        try:
            with open(order_file_path, 'r') as f: orders = json.load(f)
            order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
            print(f"    -> Órdenes cargadas para {original_city_name}: {order_tuple}{seasonal_order_tuple}")
        except Exception:
            print(f"    -> Órdenes no encontradas para {city_name_safe}. Usando (1,1,1)(1,1,0,24).")
            order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Procesando escenario: {scenario_folder}")
            
            # SOLUCIÓN: Limitar el número de repeticiones a procesar
            all_files = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))
            files_to_process = all_files[:N_REPETITIONS_TO_PROCESS]
            
            if not files_to_process:
                print(f"      -> No se encontraron archivos para procesar en este escenario.")
                continue

            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                print(f"      -> Procesando archivo: {file_name}")
                rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                try:
                    df_rep = pd.read_csv(rep_file_path)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                    df_rep['Hour'] = df_rep['Date'].dt.hour
                    df_rep['DayOfWeek'] = df_rep['Date'].dt.dayofweek
                    df_rep['Month'] = df_rep['Date'].dt.month
                except Exception as e:
                    print(f"        -> Error cargando {file_name}: {e}. Omitiendo.")
                    continue
                
                if df_rep.isnull().values.any():
                    print(f"        -> ADVERTENCIA: NaNs en {file_name}. Rellenando con ffill/bfill.")
                    df_rep = df_rep.ffill().bfill()
                    if df_rep.isnull().values.any(): continue
                
                horizon_metrics_rep = train_predict_eval_sarimax_multi_horizon(
                    df_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
                    order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS
                )
                for hm_rep in horizon_metrics_rep:
                    all_prediction_metrics_detailed.append({
                        'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': hm_rep['Horizon'],
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\n[OK] Todas las métricas detalladas guardadas en: {detailed_output_path}")

    df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(
        Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
        Count=('MAE', 'count')
    ).reset_index()
    summary_by_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_summary_by_city.to_csv(summary_by_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_by_city_path}")

    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(
        Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
    print("\n--- Resumen Global por Método y Horizonte ---")
    print(df_global_summary)

    df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(
        Final_Avg_MAE=('Global_Avg_MAE', 'mean'),
        Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'),
        Final_Avg_MASE=('Global_Avg_MASE', 'mean')
    ).reset_index()
    avg_over_horizons_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    df_avg_over_horizons.to_csv(avg_over_horizons_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {avg_over_horizons_path}")
    print("\n--- Resumen Final por Método (promedio de todas las ciudades, escenarios y horizontes) ---")
    print(df_avg_over_horizons)

script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

DataFrame original 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

ATENCIÓN: Se procesará únicamente la ciudad: Brasilia

--- Predicción SARIMAX con Datos Originales (Baseline) ---
  Procesando ciudad (Baseline): Brasilia
    -> Órdenes cargadas para Brasilia: (1, 0, 0)(1, 0, 1, 24)
      -> Iniciando validación cruzada con 2 fold(s)...
        -> Fold 1/2...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
        -> Fold 2/2...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
    -> Finalizado Baseline para Brasilia.

--- Predicción SARIMAX con Datos Imputados ---

--- Procesando método de imputación: interpolacionLineal ---
  Procesando ciudad: Brasilia
    -> Órdenes cargadas para Brasilia: (1, 0, 0)(1, 0, 1, 24)
    Procesando escenario: BlockRows_12h_20percent
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep1_seed2510790048_imputed_interpolacionLineal.cs

KeyboardInterrupt: 